## Level 1 — Reading Data

- Task 1: Read CSV file

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/new/phase2csv/DataPhaseII.csv",
    header=True,
    inferSchema=True
)
 
df.show()

+--------+-------------+---------+-------+-------+----------+
|order_id|customer_name|     city|product|revenue|order_date|
+--------+-------------+---------+-------+-------+----------+
|     101|        Arjun|    Kochi| Laptop|  50000|01-01-2024|
|     102|        Meera|  Chennai| Mobile|  20000|03-01-2024|
|     103|        Rahul|Bangalore| Tablet|  30000|05-01-2024|
|     104|        Sneha|    Kochi|     TV|  60000|07-01-2024|
|     105|       Vikram|    Delhi| Laptop|   NULL|09-01-2024|
|     106|         Asha|  Chennai| Mobile|  25000|11-01-2024|
|     106|         Asha|  Chennai| Mobile|  25000|11-01-2024|
|    NULL|         NULL|     NULL|   NULL|   NULL|      NULL|
|    NULL|         NULL|     NULL|   NULL|   NULL|      NULL|
+--------+-------------+---------+-------+-------+----------+



- Task 2: Show schema

In [0]:
df.printSchema()


root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: string (nullable = true)



- Task 3: Convert order_date to date type

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("order_date", to_date(df.order_date, "dd-MM-yyyy"))
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)



In [0]:
df = df.withColumn("revenue", col("revenue").cast("int"))
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)



## Level 2 — Validation

- Task 4: Find rows with null revenue

In [0]:
from pyspark.sql.functions import *
df.filter(isnull("revenue")).show()

+--------+-------------+-----+-------+-------+----------+
|order_id|customer_name| city|product|revenue|order_date|
+--------+-------------+-----+-------+-------+----------+
|     105|       Vikram|Delhi| Laptop|   NULL|2024-01-09|
|    NULL|         NULL| NULL|   NULL|   NULL|      NULL|
|    NULL|         NULL| NULL|   NULL|   NULL|      NULL|
+--------+-------------+-----+-------+-------+----------+



- Task 5: Count duplicate rows

In [0]:
dup = df.groupBy(df.columns).count().filter("count > 1")
dup.show()
dup.count()

+--------+-------------+-------+-------+-------+----------+-----+
|order_id|customer_name|   city|product|revenue|order_date|count|
+--------+-------------+-------+-------+-------+----------+-----+
|     106|         Asha|Chennai| Mobile|  25000|2024-01-11|    2|
|    NULL|         NULL|   NULL|   NULL|   NULL|      NULL|    2|
+--------+-------------+-------+-------+-------+----------+-----+



2

- Task 6: Remove duplicates

In [0]:
df = df.dropDuplicates()
df.show()

+--------+-------------+---------+-------+-------+----------+
|order_id|customer_name|     city|product|revenue|order_date|
+--------+-------------+---------+-------+-------+----------+
|     101|        Arjun|    Kochi| Laptop|  50000|2024-01-01|
|     104|        Sneha|    Kochi|     TV|  60000|2024-01-07|
|     102|        Meera|  Chennai| Mobile|  20000|2024-01-03|
|     106|         Asha|  Chennai| Mobile|  25000|2024-01-11|
|     103|        Rahul|Bangalore| Tablet|  30000|2024-01-05|
|    NULL|         NULL|     NULL|   NULL|   NULL|      NULL|
|     105|       Vikram|    Delhi| Laptop|   NULL|2024-01-09|
+--------+-------------+---------+-------+-------+----------+



In [0]:
df = df.na.drop()
df.show()


+--------+-------------+---------+-------+-------+----------+
|order_id|customer_name|     city|product|revenue|order_date|
+--------+-------------+---------+-------+-------+----------+
|     101|        Arjun|    Kochi| Laptop|  50000|2024-01-01|
|     104|        Sneha|    Kochi|     TV|  60000|2024-01-07|
|     102|        Meera|  Chennai| Mobile|  20000|2024-01-03|
|     106|         Asha|  Chennai| Mobile|  25000|2024-01-11|
|     103|        Rahul|Bangalore| Tablet|  30000|2024-01-05|
+--------+-------------+---------+-------+-------+----------+



## Level 3 — Transformations
- Task 7: Create: revenue_category
    - <30000 → Low
    - 30000–50000 → Medium
    - 50000 → High

In [0]:
from pyspark.sql.functions import when, col

# Create revenue_category column
df = df.withColumn(
    "revenue_category",
    when(col("revenue") < 30000, "Low")
    .when((col("revenue") >= 30000) & (col("revenue") < 50000), "Medium")
    .otherwise("High")
)

# Display result
df.show()

+--------+-------------+---------+-------+-------+----------+----------------+
|order_id|customer_name|     city|product|revenue|order_date|revenue_category|
+--------+-------------+---------+-------+-------+----------+----------------+
|     101|        Arjun|    Kochi| Laptop|  50000|2024-01-01|            High|
|     104|        Sneha|    Kochi|     TV|  60000|2024-01-07|            High|
|     102|        Meera|  Chennai| Mobile|  20000|2024-01-03|             Low|
|     106|         Asha|  Chennai| Mobile|  25000|2024-01-11|             Low|
|     103|        Rahul|Bangalore| Tablet|  30000|2024-01-05|          Medium|
+--------+-------------+---------+-------+-------+----------+----------------+



- Task 8: Find: total revenue per city

In [0]:
df.groupBy("city").agg(sum("revenue").alias("total_revenue")).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|    Kochi|       110000|
|  Chennai|        45000|
|Bangalore|        30000|
+---------+-------------+



In [0]:
df = df.withColumn("year", year("order_date"))
df.show()

+--------+-------------+---------+-------+-------+----------+----------------+----+
|order_id|customer_name|     city|product|revenue|order_date|revenue_category|year|
+--------+-------------+---------+-------+-------+----------+----------------+----+
|     101|        Arjun|    Kochi| Laptop|  50000|2024-01-01|            High|2024|
|     104|        Sneha|    Kochi|     TV|  60000|2024-01-07|            High|2024|
|     102|        Meera|  Chennai| Mobile|  20000|2024-01-03|             Low|2024|
|     106|         Asha|  Chennai| Mobile|  25000|2024-01-11|             Low|2024|
|     103|        Rahul|Bangalore| Tablet|  30000|2024-01-05|          Medium|2024|
+--------+-------------+---------+-------+-------+----------+----------------+----+



## Level 4 — Writing Output
- Task 9: Save cleaned data as:
    - CSV
    OR
    - table

In [0]:
# Save as CSV
df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/new/phase2/")



In [0]:
# Save as another new table
df.write.mode("overwrite").saveAsTable("cleaned_df_21")